# Sausage Links #

Реализация алгоритма Sausage Links на базе Swinging Door на Python.

Литература:
 * [Патент US4669097](https://patents.google.com/patent/US4669097 "patents.google.com");
 * [Swinging Door Trending Compression Algorithm for IoT Environments](https://sol.sbc.org.br/index.php/sbesc_estendido/article/download/8650/8551/#:~:text=The%20Swing%20Door%20Trending%20(SDT,process%20information%20systems%20(PIMs). "sol.sbc.org.br");
 * [Компрессия данных в системах промышленной автоматизации. Алгоритм SwingingDoor](https://habr.com/ru/post/105652/ "habr.com");

Реализация:

In [1]:
def _sloping_calc(stretch, deviation):
    current, entrance = stretch

    if isinstance(deviation, tuple):
        dev_up, dev_down = deviation

    else:
        dev_up = dev_down = deviation

    dx = current[0] - entrance[0]

    if not dx:
        raise ValueError(
            "The division by 0 occurs during the calculation of the slope."
        )

    upper = (current[1] - (entrance[1] + dev_up)) / dx
    lower = (current[1] - (entrance[1] - dev_down)) / dx

    return upper, lower


def sausage_links(
    source,
    deviation=0.1,
    max_len=0,
    auto_dev_factor=0,
    ema_alpha=0.3,
):
    if not deviation and not auto_dev_factor:
        yield from source
        return

    try:
        entrance = next(source)

    except StopIteration:
        return

    yield entrance

    try:
        current = next(source)

    except StopIteration:
        return

    if auto_dev_factor:
        dx = current[0] - entrance[0]
        ema_slope = (
            (abs(current[1] - entrance[1]) / dx) if dx else 0.0
        )

        current_deviation = auto_dev_factor * ema_slope

    else:
        current_deviation = deviation

    sloping_upper, sloping_lower = _sloping_calc(
        (current, entrance), current_deviation
    )

    sloping_upper_max = sloping_upper
    sloping_lower_min = sloping_lower

    try:
        while True:
            past = current
            current = next(source)

            if auto_dev_factor:
                dx = current[0] - past[0]
                ema_slope = (
                    ema_alpha
                    * ((abs(current[1] - past[1]) / dx) if dx else 0.0)
                    + (1 - ema_alpha) * ema_slope
                )

                current_deviation = auto_dev_factor * ema_slope

            if (
                max_len > 0
                and abs(current[0] - entrance[0]) >= max_len
            ):
                yield past

                entrance = current
                yield entrance

                try:
                    current = next(source)

                except StopIteration:
                    return

                sloping_upper, sloping_lower = _sloping_calc(
                    (current, entrance), current_deviation
                )
                sloping_upper_max = sloping_upper
                sloping_lower_min = sloping_lower
                continue

            sloping_upper, sloping_lower = _sloping_calc(
                (current, entrance), current_deviation
            )

            if sloping_upper > sloping_upper_max:
                sloping_upper_max = sloping_upper

                if sloping_upper_max > sloping_lower_min:
                    yield past

                    entrance = current
                    yield entrance

                    try:
                        current = next(source)

                    except StopIteration:
                        return

                    if auto_dev_factor:
                        dx = current[0] - entrance[0]
                        ema_slope = (
                            ema_alpha
                            * (
                                (abs(current[1] - entrance[1]) / dx)
                                if dx
                                else 0.0
                            )
                            + (1 - ema_alpha) * ema_slope
                        )

                        current_deviation = auto_dev_factor * ema_slope

                    sloping_upper_max, sloping_lower_min = _sloping_calc(
                        (current, entrance), current_deviation
                    )

            elif sloping_lower < sloping_lower_min:
                sloping_lower_min = sloping_lower

                if sloping_upper_max > sloping_lower_min:
                    yield past

                    entrance = current
                    yield entrance

                    try:
                        current = next(source)

                    except StopIteration:
                        return

                    if auto_dev_factor:
                        dx = current[0] - entrance[0]
                        ema_slope = (
                            ema_alpha
                            * (
                                (abs(current[1] - entrance[1]) / dx)
                                if dx
                                else 0.0
                            )
                            + (1 - ema_alpha) * ema_slope
                        )

                        current_deviation = auto_dev_factor * ema_slope

                    sloping_upper_max, sloping_lower_min = _sloping_calc(
                        (current, entrance), current_deviation
                    )

    except StopIteration:
        yield past


Тестирование:

In [2]:
assert list(sausage_links(iter([
    (1, 6), (2, 6.5), (3, 5.5),
    (4, 6.5), (5, 8), (6, 7.5),
    (7, 8), (8, 9.5),
]), 1)) == [(1, 6), (7, 8), (8, 9.5)]

In [3]:
assert list(sausage_links(iter([
    (1, 6), (2, 6.5), (3, 5.5),
    (4, 6.5), (5, 8), (6, 7.5),
    (7, 8), (8, 6),
]), 1)) == [(1, 6), (7, 8), (8, 6)]

Проверка:

In [4]:
from matplotlib.pyplot import plot, show
from ipywidgets import FloatSlider, interact, IntSlider

my_data = [
    (0., 5.0), (1., 5.5), (2., 4.2),
    (3., 5.8), (4., 5.2), (5., 6.8),
]

def re_show(deviation_up, deviation_down, max_len, auto_dev_factor, ema_alpha):
    o_data = list()
    o_index = list()
    
    for x, y in my_data:
        o_data.append(y)
        o_index.append(x)
    
    n_data = list()
    n_index = list()

    for x, y in sausage_links(iter(my_data), (deviation_up, deviation_down), max_len, auto_dev_factor, ema_alpha):
        print(x, y)
        n_data.append(y)
        n_index.append(x)

    plot(o_index, o_data)
    plot(n_index, n_data)

    show()
    
interact(
    re_show,
    deviation_up=FloatSlider(min=.01, max=10, step=.01, value=1),
    deviation_down=FloatSlider(min=.01, max=10, step=.01, value=1),
    max_len=IntSlider(min=0, max=100, step=1, value=0),
    auto_dev_factor=FloatSlider(min=0, max=10, step=.1, value=0),
    ema_alpha=FloatSlider(min=0, max=1, step=.1, value=.3)
);

interactive(children=(FloatSlider(value=1.0, description='deviation_up', max=10.0, min=0.01, step=0.01), Float…

---

## Применение

Постройка графика из сырых данных:

In [5]:
from datetime import datetime

from pandas import read_csv
from matplotlib.pyplot import figure, show
from ipywidgets import FloatSlider, interact, IntSlider

from sausage_links import sausage_links

data = [
    (datetime.strptime(date, "%Y-%m-%d").timestamp(), value)
    for date, value in read_csv(
        "https://raw.githubusercontent.com/datasets/oil-prices/refs/heads/main/data/wti-daily.csv"
    ).values.tolist()
]

def re_show(deviation_up, deviation_down, max_len, auto_dev_factor, ema_alpha):
    x = list()
    y = list()
    
    for k, v in data:
        x.append(datetime.fromtimestamp(k))
        y.append(v)

    s_x = list()
    s_y = list()

    s_data = list(
        sausage_links(iter(data), (deviation_up, deviation_down), max_len * 60 * 60 * 24, auto_dev_factor, ema_alpha)
    )

    for k, v in s_data:
        s_x.append(datetime.fromtimestamp(k))
        s_y.append(v)

    fig = figure()
    fig.set_dpi(100)
    fig.suptitle("The operation of the Swinging Door algorithm.")

    gs = fig.add_gridspec(2, hspace=.3)

    (ax_orig, ax_sd) = fig.add_gridspec(2, hspace=.3).subplots(sharex=True, sharey=True)

    ax_orig.set_title("Original data")
    ax_orig.plot(x, y, "tab:blue", label=f"count: {len(data)}")
    ax_orig.legend(loc="upper left")

    ax_sd.set_title(f"Compressed data use deviation {deviation_up}, {deviation_down}")
    ax_sd.plot(s_x, s_y, "tab:orange", label=f"count: {len(s_data)}")
    ax_sd.legend(loc="upper left")

    show()
    
interact(
    re_show,
    deviation_up=FloatSlider(min=.01, max=10, step=.01, value=1),
    deviation_down=FloatSlider(min=.01, max=10, step=.01, value=1),
    max_len=IntSlider(min=0, max=100, step=1, value=0),
    auto_dev_factor=FloatSlider(min=0, max=1000000, step=1000, value=0),
    ema_alpha=FloatSlider(min=0, max=1, step=.1, value=.3)
);

interactive(children=(FloatSlider(value=1.0, description='deviation_up', max=10.0, min=0.01, step=0.01), Float…